In [1]:
# conda env 'vis2'

import sys
import os
sys.path.append('/data/keeling/a/johntl3/b/LundstromPub2025/src')

from util import load_sensor_dict
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import glob
import at3d

from collections import defaultdict

In [ ]:
def parse_at3d_output(dir):
    loss_file = dir + '/loss.csv'
    log_file = dir + '/at3d.log'
    info = {}
    try:
        with open(log_file, 'r') as file:
            lines = list(file)
            info['termination'] = lines[-1].strip()
            info['F'] = float(lines[-3].split()[2])
            info['N'] = int(lines[-4].split()[0])
            info['Tit'] = int(lines[-4].split()[1])
            info['Tnf'] = int(lines[-4].split()[2])
            info['Tnint'] =  int(lines[-4].split()[3])
            info['Skip'] = int(lines[-4].split()[4])
            info['Nact'] = int(lines[-4].split()[5])
            info['Projg'] = float(lines[-4].split()[6].replace('D','E'))
            info['max_cell_splitting'] = False
            for l in lines:
                if('Further cell splitting' in l):
                    info['max_cell_splitting'] = True
                if('Execution time' in l):
                    info['run time'] = float(l.split()[2])
    except FileNotFoundError:
        print('Error: File not found at {}'.format(log_file))
    except Exception as e:
        print(f"Error while parsing log file: {e}")

    try:
        with open(loss_file, 'r') as file:
            lines = list(file)
            info['total_obj_fn_time'] = float(lines[-1].split()[2])
    except FileNotFoundError:
        print('Error: File not found at {}'.format(loss_file))
    except Exception as e:
        print(f"Error while parsing log file: {e}")
    
    return info

In [5]:
def load_retreival(dir):
    assert os.path.isdir(dir + '/iterations/')
    name = os.path.split(dir)[-1]
    files = np.sort(glob.glob(dir + '/iterations/iter*.nc'))
    sensors, solvers, rte_grid = at3d.util.load_forward_model(files[-1])
    data = {
        'files': files,
        'sensors': sensors,
        'solvers': solvers,
        'rte_grid': rte_grid,
        'log': parse_at3d_output(dir),
    }
    return data

In [6]:
def traverse(dir, data, parent_key=()):
    name = os.path.split(dir)[-1]
    parent_key += (name,)
    if(os.path.isdir(dir + '/iterations/')):
        data[parent_key] = load_retreival(dir)
    else:
        for n in os.listdir(dir):
            traverse(dir+'/'+n, data, parent_key)


In [7]:
version_dir = '/data/keeling/a/johntl3/b/LundstromPub2025/CAMP2Ex_RF17/AT3D_output/V02'
data = defaultdict(defaultdict)
traverse(version_dir, data)

/data/keeling/a/johntl3/miniconda3/envs/vis2/lib/python3.10/site-packages/xarray/core/concat.py:500: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))
/data/keeling/a/johntl3/miniconda3/envs/vis2/lib/python3.10/site-packages/at3d/solver.py:2337: UserWarning: Number of SHDOM grid cells with optical depth greater than 2: '59'. Max cell optical depth: '3.2738821506500244'
  warnings.warn("Number of SHDOM grid cells with optical depth greater than 2: '{}'. "
/data/keeling/a/johntl3/miniconda3/envs/vis2/lib/python3.10/site-packages/xarray/core/concat.py:500: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))
/data/keeling/a/johntl3/miniconda3/envs/vis2/lib/python3

In [8]:
# def print_nested_dictionary_keys(nested_dict, indent=0):
#     """
#     Recursively prints keys from a nested dictionary with indentation.

#     Args:
#         nested_dict (dict): The nested dictionary to print keys from.
#         indent (int): The current indentation level.
#     """
#     for key, value in nested_dict.items():
#         print('  ' * indent + str(key)) # Print key with indentation
#         if isinstance(value, dict):
#             print_nested_dictionary_keys(value, indent + 1) # Recursive call for nested dictionaries

# print_nested_dictionary_keys(data)

### Error Metrics

$\bm{\beta_p}$ is the perturbed extinction coefficient vector. $\bm{\beta_r}$ is the reference extinction coefficient vector.

Relative Bias (Jesse's relative bias):

$\text{Relative Bias} = 100 \times \frac{||\bm{\beta_p}||_1 - ||\bm{\beta_r}||_1}{||\bm{\beta_r}||_1}$

Relative Difference (Jesse's relative RMSE):

$\text{Relative Difference} = 100 \times \frac{||\bm{\beta_p} - \bm{\beta_r}||_2}{||\bm{\beta_r}||_2}$

$||\bm{x}||_1 = \sum_i |x_i|$ and $||\bm{x}||_2 = \left(\sum_i x_i^2\right)^{\frac{1}{2}}$

Mean Absolute Error:

$\text{MAE} = \frac{1}{n} \sum_{i=1}^n |\beta_{p,i} - \beta_{r, i}|$

Root Mean Square Error:

$\text{RMSE} = \left(\frac{1}{n} \sum_{i=1}^n \left(\beta_{p,i} - \beta_{r,i}\right)^2\right)^\frac{1}{2} = \left(\frac{1}{n} ||\bm{\beta_p} - \bm{\beta_r}||_2^2\right)^{\frac{1}{2}}$

Correlation Coefficient (Pearson's r):

$r = \frac{\sum_i (\beta_{p,i} - \bar{\bm{\beta_p}})(\beta_{r,i} - \bar{\bm{\beta_r}})}{\sqrt{\sum_i (\beta_{p,i} - \bar{\bm{\beta_p}})^2} \sqrt{\sum_i (\beta_{r,i} - \bar{\bm{\beta_r}})^2}}$

Normalized Mean Bias (NMB):

$\text{NMB} = \frac{\sum_i (\beta_{p,i} - \beta_{r,i})}{\sum_i \beta_{r,i}}$


In [9]:
def calc_error_metrics(reference, data):
    if('solvers' in list(data.keys())):
        ext_tmp = np.ravel(data['solvers'][0.67].medium['cloud'].extinction.data)
        ext_ref = np.ravel(reference['solvers'][0.67].medium['cloud'].extinction.data)
        data['relative difference'] = 100 * (np.sum((ext_tmp - ext_ref)**2))**0.5 / (np.sum(ext_ref)**2)**0.5
        data['relative bias'] = 100 * (np.sum(np.abs(ext_tmp)) - np.sum(np.abs(ext_ref))) / (np.sum(np.abs(ext_ref)))
        data['MAE'] = np.sum(np.abs(ext_tmp - ext_ref)) / len(ext_tmp)
        data['RMSE'] = (np.sum((ext_tmp - ext_ref)**2) / len(ext_tmp))**0.5
        data['r'] = (np.sum((ext_tmp - np.mean(ext_tmp)) * (ext_ref - np.mean(ext_ref)))) / ((np.sum((ext_tmp - np.mean(ext_tmp))**2)**0.5) * (np.sum((ext_ref - np.mean(ext_ref))**2)**0.5))
        data['NMB'] = np.sum(ext_tmp - ext_ref) / np.sum(ext_ref)

        tmp = data['solvers'][0.67].medium['cloud'].copy(deep=True)
        mask = tmp.extinction.data > 0
        tmp['cvm'] = (['x', 'y', 'z'], mask)
        cv = tmp.cvm.integrate(['x','y','z']).data
        tmp_ref = reference['solvers'][0.67].medium['cloud'].copy(deep=True)
        mask = tmp_ref.extinction.data > 0
        tmp_ref['cvm'] = (['x', 'y', 'z'], mask)
        cv_ref = tmp_ref.cvm.integrate(['x','y','z']).data
        data['cloud volume'] = cv
        data['delta volume'] = cv - cv_ref
    else:
        for n in data:
            calc_error_metrics(reference, data[n])

In [10]:
reference = data[('V02', 'default')]
calc_error_metrics(reference, data)

In [11]:
CVM_file = '/data/keeling/a/johntl3/b/LundstromPub2025/CAMP2Ex_RF17/intermediate_data/CVM_default_open.nc'
cvm = xr.load_dataset(CVM_file)
vol_cvm = cvm.cloud_volume_mask.integrate(['x','y','z']).data
print('CVM Volume: {:.3f} km**3'.format(vol_cvm))

CVM Volume: 76.179 km**3


In [12]:
print('{:<48} | {:<19} | {:<23} | {:<20} | {:<20}'.format('Path', 'Relative Bias (%)', 'Relative Difference (%)', 'Cloud Volume (km**3)', 'Change in Volume (km**3)'))
print('-'*148)
for k, v in data.items():
    print('{:<48} | {:<19} | {:<23} | {:<20} | {:<20}'.format(str(k), v['relative bias'], v['relative difference'], v['cloud volume'], v['delta volume']))

Path                                             | Relative Bias (%)   | Relative Difference (%) | Cloud Volume (km**3) | Change in Volume (km**3)
----------------------------------------------------------------------------------------------------------------------------------------------------
('V02', 'default')                               | 0.0                 | 0.0                     | 24.826553125         | 0.0                 
('V02', 'CVM', 'unconstrained')                  | 42.54360330432428   | 2.2331556357898155      | 47.5070578125        | 22.6805046875       
('V02', 'CVM', 'unconstrained_darksurf')         | 67.5925064079974    | 2.4514915532464148      | 64.80835625          | 39.981803125        
('V02', 'CVM', 'LCL')                            | 48.31044134878878   | 2.7553765986789216      | 36.9864859375        | 12.1599328125       
('V02', 'CVM', 'CTH')                            | -5.011142936837676  | 1.0900163613522225      | 43.8018109375        | 18.9752578

In [14]:
print('{:<48} | {:<19} | {:<23} | {:<20} | {:<20}'.format('Path', 'F', 'tot obj fcn time', 'Max Cell Splitting', 'Termination'))
print('-'*148)
for k, v in data.items():
    print('{:<48} | {:<19} | {:<23} | {:<20} | {:<20}'.format(str(k), v['log']['F'], v['log']['total_obj_fn_time'], v['log']['max_cell_splitting'], v['log']['termination']))

Path                                             | F                   | tot obj fcn time        | Max Cell Splitting   | Termination         
----------------------------------------------------------------------------------------------------------------------------------------------------
('V02', 'default')                               | 648.8699401832733   | 8077.9941472790015      | 0                    | CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
('V02', 'CVM', 'unconstrained')                  | 620.8244892587431   | 15367.394563017002      | 0                    | CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
('V02', 'CVM', 'unconstrained_darksurf')         | 666.4566918489049   | 9989.371398908          | 0                    | ABNORMAL_TERMINATION_IN_LNSRCH
('V02', 'CVM', 'LCL')                            | 630.2697839295879   | 15846.239144664007      | 0                    | CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
('V02', 'CVM', 'CTH')                        